# SaaS Customer Churn Prediction

## Project Overview
This project analyzes SaaS subscription data to identify customers at risk of churn and build machine learning models for churn prediction.

The final workflow uses a **time-based churn definition**: customer features are created from information available before the prediction cutoff, while churn is determined from behavior after the cutoff.

### Goals
- Understand customer and subscription behavior
- Define churn using a future observation window
- Build customer-level features
- Train and compare classification models
- Identify high-risk customers for retention analysis

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

## 2. Load the Dataset

In [ ]:
file_path = "saas-subscriptions-v1.0.0-analysis.xlsx"

excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

In [ ]:
df = pd.read_excel(
    file_path,
    sheet_name="SaaS Subscriptions & Customer C"
)

df["invoice_date"] = pd.to_datetime(df["invoice_date"])

print("Shape:", df.shape)
df.head()

## 3. Data Understanding

In [ ]:
df.info()

print("\nMissing values:")
print(df.isna().sum())

In [ ]:
print("Date range:", df["invoice_date"].min(), "to", df["invoice_date"].max())
print("Unique customers:", df["account_id"].nunique())
print("Unique subscriptions:", df["subscription_id"].nunique())

print("\nInvoice status:")
print(df["invoice_status"].value_counts())

print("\nPlan:")
print(df["plan_name"].value_counts())

print("\nSegment:")
print(df["segment"].value_counts())

## 4. Define Churn

A customer is included in the prediction cohort if they had invoice activity during the **60 days before the cutoff date**.

A customer is labeled as **churned** if they have no invoice activity during the **90 days after the cutoff date**.

This separates the information used to create features from the future period used to define the target.

In [ ]:
cutoff_date = pd.Timestamp("2025-07-01")
active_window_start = cutoff_date - pd.Timedelta(days=60)
future_end = cutoff_date + pd.Timedelta(days=90)

past_df = df[df["invoice_date"] < cutoff_date].copy()
future_df = df[df["invoice_date"] >= cutoff_date].copy()

print("Cutoff:", cutoff_date.date())
print("Active window starts:", active_window_start.date())
print("Future window ends:", future_end.date())
print("Past rows:", len(past_df))
print("Future rows:", len(future_df))

In [ ]:
active_customers = past_df.loc[
    past_df["invoice_date"] >= active_window_start, "account_id"
].unique()

future_active_customers = future_df.loc[
    future_df["invoice_date"] < future_end, "account_id"
].unique()

continued_customers = set(active_customers) & set(future_active_customers)
churned_customers = set(active_customers) - continued_customers

print("Active customers:", len(active_customers))
print("Continued customers:", len(continued_customers))
print("Churned customers:", len(churned_customers))

## 5. Customer-Level Feature Engineering

The invoice-level data is aggregated to one row per customer. All predictive features below come from data available before the cutoff date.

In [ ]:
model_df = past_df[past_df["account_id"].isin(active_customers)].copy()

customer_model = model_df.groupby("account_id").agg(
    first_invoice_date=("invoice_date", "min"),
    last_invoice_date=("invoice_date", "max"),
    invoice_count=("invoice_id", "count"),
    total_mrr=("mrr", "sum"),
    avg_mrr=("mrr", "mean"),
    total_amount_paid=("amount_paid", "sum"),
    avg_amount_paid=("amount_paid", "mean"),
    total_seats=("seats", "sum"),
    avg_seats=("seats", "mean")
).reset_index()

customer_model["days_inactive_at_cutoff"] = (
    cutoff_date - customer_model["last_invoice_date"]
).dt.days

customer_model["tenure"] = (
    cutoff_date - customer_model["first_invoice_date"]
).dt.days

In [ ]:
customer_info_model = model_df.groupby("account_id").agg(
    plan_name=("plan_name", "last"),
    region=("region", "last"),
    segment=("segment", "last")
).reset_index()

customer_model = customer_model.merge(
    customer_info_model,
    on="account_id",
    how="left"
)

customer_model["region"] = customer_model["region"].fillna("unknown")

customer_model["churn"] = (
    customer_model["account_id"].isin(churned_customers)
).astype(int)

print("Customer-level shape:", customer_model.shape)
print(customer_model["churn"].value_counts())
print("Churn rate:", round(customer_model["churn"].mean() * 100, 2), "%")

## 6. Exploratory Churn Analysis

In [ ]:
plan_churn = (
    customer_model.groupby("plan_name")["churn"]
    .mean().mul(100).reset_index(name="churn_rate")
)
plan_churn

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(plan_churn["plan_name"], plan_churn["churn_rate"])
plt.title("Churn Rate by Subscription Plan")
plt.xlabel("Subscription Plan")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
inactivity_churn = (
    customer_model.groupby("churn")["days_inactive_at_cutoff"]
    .mean().reset_index()
)
inactivity_churn["status"] = inactivity_churn["churn"].map({0: "Retained", 1: "Churned"})
inactivity_churn

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(inactivity_churn["status"], inactivity_churn["days_inactive_at_cutoff"])
plt.title("Average Customer Inactivity by Churn Status")
plt.xlabel("Customer Status")
plt.ylabel("Average Days Inactive")
plt.tight_layout()
plt.show()

In [ ]:
segment_churn = customer_model.groupby("segment")["churn"].mean().mul(100).reset_index(name="churn_rate")
region_churn = customer_model.groupby("region")["churn"].mean().mul(100).reset_index(name="churn_rate")

print("Churn by segment:")
display(segment_churn)
print("Churn by region:")
display(region_churn)

## 7. Prepare Data for Machine Learning

In [ ]:
features = [
    "invoice_count", "tenure", "total_mrr", "avg_mrr",
    "total_amount_paid", "avg_amount_paid", "total_seats",
    "avg_seats", "days_inactive_at_cutoff", "plan_name",
    "region", "segment"
]

X = customer_model[features]
y = customer_model["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X:", X.shape)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

In [ ]:
categorical_features = ["plan_name", "region", "segment"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

## 8. Train Classification Models

Three classifiers are compared. `class_weight="balanced"` gives more weight to the minority churn class.

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"))
])

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42,
        n_jobs=-1, class_weight="balanced"
    ))
])

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Random Forest": rf_model
}

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    print("=" * 60)
    print(name)
    print("=" * 60)
    print(confusion_matrix(y_test, predictions))
    print(classification_report(y_test, predictions, zero_division=0))

## 9. Cross-Validation Model Comparison

A single train/test split can be unstable because the cohort contains only 82 churned customers. Five-fold cross-validation gives a broader view of performance. F1 is used because it balances precision and recall for the churn class.

In [ ]:
cv_results = []

for name, model in models.items():
    scores = cross_validate(
        model, X, y, cv=5,
        scoring={"precision": "precision", "recall": "recall", "f1": "f1"},
        zero_division=0
    )
    cv_results.append({
        "model": name,
        "mean_precision": scores["test_precision"].mean(),
        "mean_recall": scores["test_recall"].mean(),
        "mean_f1": scores["test_f1"].mean()
    })

cv_results = pd.DataFrame(cv_results).sort_values("mean_f1", ascending=False)
cv_results

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(cv_results["model"], cv_results["mean_f1"])
plt.title("5-Fold Cross-Validation F1 Score")
plt.xlabel("Model")
plt.ylabel("Mean F1 Score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 10. Random Forest Feature Importance

These are impurity-based feature importances from the trained Random Forest. They describe model usage, not causation.

In [ ]:
rf_classifier = rf_model.named_steps["classifier"]
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_classifier.feature_importances_
})

feature_importance["feature"] = (
    feature_importance["feature"]
    .str.replace("cat__", "", regex=False)
    .str.replace("remainder__", "", regex=False)
)

feature_importance = feature_importance.sort_values("importance", ascending=False)
feature_importance.head(15)

In [ ]:
top_features = feature_importance.head(10).sort_values("importance")

plt.figure(figsize=(9, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 11. High-Risk Customer Identification

Out-of-fold probabilities are used instead of in-sample probabilities so the ranking is less optimistic. The result is a prioritization list for retention analysis, not a guarantee of individual churn.

In [ ]:
oof_probabilities = cross_val_predict(
    rf_model, X, y, cv=5, method="predict_proba"
)[:, 1]

customer_predictions = customer_model[[
    "account_id", "plan_name", "region", "segment",
    "days_inactive_at_cutoff"
]].copy()

customer_predictions["churn_probability"] = oof_probabilities
customer_predictions = customer_predictions.sort_values(
    "churn_probability", ascending=False
)

customer_predictions.head(10)

In [ ]:
top_10 = customer_predictions.head(10)
actual_top_10 = customer_model[customer_model["account_id"].isin(top_10["account_id"])]

precision_at_10 = actual_top_10["churn"].sum() / 10
recall_at_10 = actual_top_10["churn"].sum() / customer_model["churn"].sum()

print("Precision@10:", round(precision_at_10 * 100, 2), "%")
print("Recall@10:", round(recall_at_10 * 100, 2), "%")

## 12. Business Insights

- Customer inactivity at the cutoff is strongly associated with the model's churn predictions.
- Churned customers in this cohort had substantially more days of inactivity at the cutoff than retained customers.
- The small number of churn cases means validation metrics can vary considerably across folds.
- Churned customers were not simply the lowest-value customers in this cohort; the churned group had higher average MRR and seats. This is a descriptive finding, not a causal conclusion.
- Plan, region, and segment show different observed churn rates, but these differences should not be interpreted as causal effects.

### Potential retention actions
1. Use inactivity as an early-warning signal.
2. Create a retention queue for customers with high predicted churn probability.
3. Prioritize limited retention resources using the probability ranking.
4. Monitor higher-value customers who also show increasing inactivity.
5. Re-evaluate the model as more customer history becomes available.

## 13. Conclusion

This project demonstrates an end-to-end SaaS churn workflow:

**Raw invoice data → customer-level features → time-based churn definition → exploratory analysis → preprocessing → machine learning → cross-validation → feature importance → high-risk customer ranking**

The Random Forest and Decision Tree show similar mean F1 performance in five-fold cross-validation, while Logistic Regression performs lower on the churn class for this dataset. Because only 82 customers are labeled as churned, the results should be treated as a portfolio/learning analysis rather than a production-ready model.